In [ ]:
import pandas as pd
from collections import defaultdict

# ── Configuración ──────────────────────────────────────────────────────────
SAMPLES = 140000

RUTA_DE_DATASETS      = "../datasets"
DATASET_TRANSACCIONES = "LI-Small_Trans.csv"
SAMPLE_TRANSACCIONES  = "transacciones_sample.csv"

# Parámetros Q4
PERIODO_INICIO  = "2022-09-01"
PERIODO_FIN     = "2022-09-05 23:59:59"
MONEDA          = "US Dollar"
EXPECTED_B      = 5
TARGET_PATTERNS = 4        # parar al encontrar esta cantidad de patrones completos
CHUNK_SIZE      = 100_000  # filas por chunk

def guardar_csv(df, nombre_archivo):
    df.to_csv(nombre_archivo, index=False)


In [2]:
# ── 1. Leer por chunks hasta encontrar TARGET_PATTERNS patrones Q4 ────────────
# sent_to[x] = set de nodos a los que x envió (USD, en período)
sent_to     = defaultdict(set)
patterns_found = {}   # (A, C) → set de B's

chunks_read = []
rows_read   = 0

dtype_spec = {
    "From Bank": "string", "Account": "string",
    "To Bank":   "string", "Account.1": "string",
}

reader = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{DATASET_TRANSACCIONES}",
    dtype=dtype_spec,
    chunksize=CHUNK_SIZE,
)

for chunk in reader:
    chunks_read.append(chunk)
    rows_read += len(chunk)

    ts   = pd.to_datetime(chunk["Timestamp"])
    mask = (
        (chunk["Payment Currency"] == MONEDA) &
        (ts >= PERIODO_INICIO) &
        (ts <= PERIODO_FIN)
    )
    df_q4_chunk = chunk[mask]

    if len(df_q4_chunk) > 0:
        froms = (df_q4_chunk["From Bank"].astype(str) + "|" + df_q4_chunk["Account"].astype(str)).tolist()
        tos   = (df_q4_chunk["To Bank"].astype(str)   + "|" + df_q4_chunk["Account.1"].astype(str)).tolist()

        nodes_updated = set()
        for f, t in zip(froms, tos):
            sent_to[f].add(t)
            nodes_updated.add(f)

        # Chequear solo nodos actualizados en este chunk
        for a in nodes_updated:
            bs = sent_to[a]
            if len(bs) == EXPECTED_B:
                common_cs = None
                for b in bs:
                    cs_b = sent_to.get(b, set())
                    common_cs = set(cs_b) if common_cs is None else common_cs & cs_b
                    if not common_cs:
                        break
                if common_cs:
                    for c in common_cs:
                        key = (a, c)
                        if key not in patterns_found:
                            patterns_found[key] = set(bs)

    if len(patterns_found) >= TARGET_PATTERNS:
        print(f"✓ {len(patterns_found)} patrones encontrados tras leer {rows_read:,} filas. Deteniendo lectura anticipada.")
        break

    if rows_read % 500_000 < CHUNK_SIZE:
        print(f"  {rows_read:,} filas leídas — patrones: {len(patterns_found)}")

print(f"\nPatrones Q4 encontrados: {len(patterns_found)}")
for (a, c), bs in patterns_found.items():
    print(f"  A={a}  →  B×{len(bs)}  →  C={c}")


  500,000 filas leídas — patrones: 0
  1,000,000 filas leídas — patrones: 1
  1,500,000 filas leídas — patrones: 1
  2,000,000 filas leídas — patrones: 1
  2,500,000 filas leídas — patrones: 1
  3,000,000 filas leídas — patrones: 1
  3,500,000 filas leídas — patrones: 1
  4,000,000 filas leídas — patrones: 1
  4,500,000 filas leídas — patrones: 1
  5,000,000 filas leídas — patrones: 1
  5,500,000 filas leídas — patrones: 1
  6,000,000 filas leídas — patrones: 1
  6,500,000 filas leídas — patrones: 1

Patrones Q4 encontrados: 1
  A=026|801A29190  →  B×5  →  C=020|800914620


In [3]:
# ── 2. Construir muestra ──────────────────────────────────────────────────────
# Cuentas involucradas en todos los patrones encontrados
all_pattern_accounts = set()
for (a, c), bs in patterns_found.items():
    all_pattern_accounts.add(a)
    all_pattern_accounts.add(c)
    all_pattern_accounts.update(bs)

print(f"Cuentas en patrones: {len(all_pattern_accounts)}")

# Separar filas obligatorias de las ya leídas
df_leido = pd.concat(chunks_read, ignore_index=True)
df_leido["_From"] = df_leido["From Bank"].astype(str) + "|" + df_leido["Account"].astype(str)
df_leido["_To"]   = df_leido["To Bank"].astype(str)   + "|" + df_leido["Account.1"].astype(str)

mask_pat       = df_leido["_From"].isin(all_pattern_accounts) | df_leido["_To"].isin(all_pattern_accounts)
df_must        = df_leido[mask_pat].drop(columns=["_From", "_To"])
df_leido_resto = df_leido[~mask_pat].drop(columns=["_From", "_To"])

print(f"Filas obligatorias (cuentas A/B/C): {len(df_must)}")
print(f"Filas del tramo leído (pool):        {len(df_leido_resto)}")

# Leer el resto del dataset sin análisis (reader continúa desde donde quedó)
print("Leyendo resto del dataset para muestreo aleatorio...")
remaining = list(reader)
df_no_leido = pd.concat(remaining, ignore_index=True) if remaining else pd.DataFrame(columns=df_must.columns)
print(f"Filas del tramo no leído:            {len(df_no_leido)}")

# Armar muestra final
faltan = SAMPLES - len(df_must)
pool   = pd.concat([df_leido_resto, df_no_leido], ignore_index=True)

if faltan > 0:
    df_relleno = pool.sample(n=min(faltan, len(pool)), random_state=42)
    df_final   = pd.concat([df_must, df_relleno]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    print(f"ADVERTENCIA: filas obligatorias ({len(df_must)}) superan el target ({SAMPLES}). Usando solo obligatorias.")
    df_final = df_must.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTotal filas en la muestra final: {len(df_final)}")
guardar_csv(df_final, f"{RUTA_DE_DATASETS}/{SAMPLE_TRANSACCIONES}")
print(f"Guardado en {RUTA_DE_DATASETS}/{SAMPLE_TRANSACCIONES}")


Cuentas en patrones: 7
Filas obligatorias (cuentas A/B/C): 214
Filas del tramo leído (pool):        6923835
Leyendo resto del dataset para muestreo aleatorio...
Filas del tramo no leído:            0

Total filas en la muestra final: 100000
Guardado en ../datasets/transacciones_sample.csv
